# W16-D6 实验 · G-07 集合门转正：活事件回放 × 突变排练 × 计数门 vs 集合门对照

与 md 的分工：md 是转正记录与设计决策（阅读材料）；本 ipynb 是**对门本身的可执行实验**——
① 解析转正门对真实 origin/main 的机读判决（活事件回放）；② 解析 selftest 六突变与 citing 生命周期；
③ 蒙特卡洛对照实验：**计数门 vs 集合门**在四类表宇宙漂移事件下的检出率（含等量换血——计数门的结构性盲区）。
数据全部真实：基线 = 今日冻结的 canonical-baseline-w39.txt（477 表），现实 = lnkcre origin/main。

In [ ]:
# ---- 环境：中文字体（TOOLS.md 标准方式）+ 路径 ----
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

import hashlib, json, subprocess, sys, tempfile
from pathlib import Path
import numpy as np

BASE = Path("/root/learning-notebooks")
SEM = BASE / "semantic-model"
GOV = SEM / "governance"
GATE = GOV / "ci" / "canonical_drift_ci.py"
BL = GOV / "canonical-baseline-w39.txt"
LNK = Path("/root/lnkcre")
W16 = BASE / "第16周"

baseline_tables = sorted(l.strip() for l in BL.read_text(encoding="utf-8").splitlines()
                         if l.strip() and not l.startswith("#"))
print(f"基线表数: {len(baseline_tables)}（应为 477）")
assert len(baseline_tables) == 477

## §1 活事件回放：转正门对真实 origin/main 的机读判决

昨日（D5）这件事靠人工 diff commit 内容发现；今天起是 `verify --json` 的秒级输出。
本节解析机读判决并断言：+7 孤儿逐表 BROKEN + 泄漏 7 + 死登记 0 + 归因单行精确 + 集合指纹口径。

In [ ]:
# ---- §1 运行转正门（真实验证，非复述）----
r = subprocess.run([sys.executable, str(GATE), "verify", "--json"],
                   capture_output=True, text=True)
out = json.loads(r.stdout)
s, res = out["summary"], out["results"]
print(f"基线 {s['baseline']['tables']} 表（fp {s['baseline']['set_sha256_16']}）"
      f" vs 现实 {s['reality']['tables']} 表（fp {s['reality']['set_sha256_16']}）@ {s['reality']['source']}")
print(f"added={s['added']} removed={s['removed']} cited={s['cited']} leaked={s['leaked']}"
      f" broken={s['broken']} -> {s['verdict']}（exit {r.returncode}）")
print()
pos = [x for x in res if x["check"] == "positive"]
for x in pos:
    attr = x["detail"].split("来源迁移 ")[-1] if "来源迁移" in x["detail"] else "(无归因)"
    print(f"  [{x['verdict']:6s}] {x['table']:30s} {attr}")

# 断言一：活事件判决与 09-18/19 记载完全一致
assert r.returncode == 1 and s["verdict"] == "RED"
assert s["added"] == 7 and s["removed"] == 0 and s["cited"] == 0 and s["leaked"] == 7
assert s["broken"] == 14 and all(x["verdict"] == "BROKEN" for x in pos)
seven = {"leasing_policies","leasing_policy_versions","leasing_policy_lifecycle",
         "price_authority_constraints","unit_pricing_batch_ops","unit_pricing_batches","unit_pricing_batch_lines"}
assert {x["table"] for x in pos} == seven
# 断言二：归因精确到单行且 side 正确（双迁移目录并集扫描）
for x in pos:
    assert "来源迁移 pg:" in x["detail"], x
mig = {x["table"]: x["detail"].split("来源迁移 ")[1] for x in pos}
assert mig["leasing_policies"] == "pg:000227_leasing_policy_unit_pricing.up.sql@L18"
assert mig["unit_pricing_batches"] == "pg:000229_unit_pricing_batch_model.up.sql@L1"
# 断言三：集合指纹对行序不敏感（打乱基线行序 -> 同指纹）
shuffled = baseline_tables.copy(); np.random.default_rng(7).shuffle(shuffled)
fp = lambda ts: hashlib.sha256("\n".join(sorted(ts)).encode()).hexdigest()[:16]
assert fp(shuffled) == s["baseline"]["set_sha256_16"]
print()
print("§1 断言全过：7 孤儿 BROKEN + 7 泄漏 + 0 死登记；归因 pg:000227/000229 单行精确；行序不敏感指纹成立")

## §2 突变排练解析 + citing 生命周期：门必须能红、能绿、被承认后转绿

六突变是 W17 验收线 2 的加强版；citing 生命周期预演 = 「+7 表若 W17 立 change 引用，今天就会转绿」的构造性证明。

In [ ]:
# ---- §2a selftest 六突变 ----
r = subprocess.run([sys.executable, str(GATE), "selftest"], capture_output=True, text=True)
print(r.stdout.strip().splitlines()[-1])
lines = [l for l in r.stdout.splitlines() if l.startswith("[")]
for l in lines:
    print(" ", l)
assert r.returncode == 0 and sum(l.startswith("[PASS]") for l in lines) == 6, "selftest 须 6/6 PASS"

# ---- §2b citing 生命周期：构造引用全部 7 表的 change -> 门应转绿 ----
with tempfile.TemporaryDirectory() as td:
    root = Path(td); chg = root / "chg-w17-first-citing"; chg.mkdir()
    seven_list = sorted(seven)
    (chg / "proposal.md").write_text(
        "# 首个 citing change（生命周期预演）\n登记：\n" +
        "\n".join(f"- `{t}`" for t in seven_list) + "\n", encoding="utf-8")
    r2 = subprocess.run([sys.executable, str(GATE), "verify", "--changes-dir", str(root),
                         "--change", "chg-w17-first-citing", "--json"],
                        capture_output=True, text=True)
    s2 = json.loads(r2.stdout)["summary"]
    print(f"citing 立案后: ok={s2['ok']} cited={s2['cited']} broken={s2['broken']}"
          f" -> {s2['verdict']}（exit {r2.returncode}）")
    assert r2.returncode == 0 and s2["verdict"] == "GREEN"
    assert s2["cited"] == 7 and s2["broken"] == 0
print("§2 断言全过：六突变 6/6；citing-action 生命周期（红->留痕->绿）构造性走通")

## §3 对照实验：计数门 vs 集合门——四类漂移事件 × 规模 k 的蒙特卡洛

Today's Question 的实验化：计数门（只比 |R| 与 |B|）与集合门（比集合关系）在
**add / drop / swap（等量换血）/ reorder（无关扰动）** 下的检出率。
基线用真实 477 表名，事件随机注入，每格 2000 次。

In [ ]:
# ---- §3 蒙特卡洛：两类门 × 四类事件 × 规模 k ----
count_gate = lambda B, R: len(R) != len(B)          # 只看计数
set_gate   = lambda B, R: R != B                    # 看集合关系（任一方向漂移即须裁决）

EVENTS = ["add(k)", "drop(k)", "swap(k)", "reorder"]
def apply_event(B, kind, k, rng):
    R = set(B)
    if kind == "add(k)":
        R |= {f"zz_event_new_{i}" for i in range(k)}
    elif kind == "drop(k)":
        R -= set(rng.choice(sorted(B), size=k, replace=False).tolist())
    elif kind == "swap(k)":   # 等量换血：计数不变
        R -= set(rng.choice(sorted(B), size=k, replace=False).tolist())
        R |= {f"zz_event_new_{i}" for i in range(k)}
    else:                      # reorder：集合不变（换序是无关扰动，两门都应沉默）
        pass
    return R

rng = np.random.default_rng(42)
KS, N = [1, 2, 3, 5, 8], 2000
B0 = set(baseline_tables)
det = {g: {e: [] for e in EVENTS} for g in ("计数门", "集合门")}
for e in EVENTS:
    for k in KS:
        c_cnt = c_set = 0
        for _ in range(N):
            R = apply_event(baseline_tables, e, k, rng)
            c_cnt += count_gate(B0, R)
            c_set += set_gate(B0, R)
        det["计数门"][e].append(c_cnt / N)
        det["集合门"][e].append(c_set / N)

mat = np.array([[np.mean(det[g][e]) for e in EVENTS] for g in ("计数门", "集合门")])
print("检出率（对 k 平均；reorder 列 = 假阳性率，应为 0）:")
print("门\\事件  " + "  ".join(f"{e:>10s}" for e in EVENTS))
for g, row in zip(("计数门", "集合门"), mat):
    print(f"{g:5s}   " + "  ".join(f"{v:>10.1%}" for v in row))

# 断言：结构差 = 计数门对 swap 全盲（任意 k 检出率 0）、集合门全捕；reorder 双零（无假阳性）
assert all(v == 0.0 for v in det["计数门"]["swap(k)"]), "计数门对等量换血应恒 0 检出"
assert all(v == 1.0 for v in det["集合门"]["swap(k)"]), "集合门对等量换血应全捕"
assert all(v == 0.0 for v in det["计数门"]["reorder"]) and all(v == 0.0 for v in det["集合门"]["reorder"])
assert all(v == 1.0 for v in det["集合门"]["add(k)"]) and all(v == 1.0 for v in det["集合门"]["drop(k)"])

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))
ax = axes[0]
im = ax.imshow(mat, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(4), EVENTS)
ax.set_yticks(range(2), ["计数门\n(只比个数)", "集合门\n(比集合关系)"])
for i in range(2):
    for j in range(4):
        ax.text(j, i, f"{mat[i, j]:.0%}", ha="center", va="center", fontsize=13,
                color="black" if 0.2 < mat[i, j] < 1 else "white")
ax.set_title("四类表宇宙漂移事件的检出率（k∈{1,2,3,5,8}，每格2000次）", fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.046)
ax2 = axes[1]
w = 0.36
ax2.bar([x - w / 2 for x in KS], [1.0] * len(KS), w, label="集合门", color="#2e9e5b")
ax2.bar([x + w / 2 for x in KS], [0.0] * len(KS), w, label="计数门", color="#e4572e")
ax2.set_xticks(KS); ax2.set_xlabel("等量换血规模 k（删 k 张 + 加 k 张新表，总数 477 不变）")
ax2.set_yticks([0, 0.5, 1.0], ["0%", "50%", "100%"]); ax2.set_ylim(0, 1.15)
ax2.legend(loc="upper center", ncol=2, frameon=False)
ax2.set_title("等量换血：计数门的结构性盲区（换血不改变总量）", fontsize=11)
plt.tight_layout(); plt.savefig(W16 / "w16d6_gate_vs_count.png", dpi=130); plt.show()
print("图已存 第16周/w16d6_gate_vs_count.png")
print("§3 断言全过：swap 计数门 0% vs 集合门 100%（全 k）；add/drop 双门 100%；reorder 双门 0%（零假阳性）")

## §4 汇总断言：今日交付物的存在性与一致性

In [ ]:
# ---- §4 交付物清单与终检 ----
deliverables = {
    "集合门": GATE,
    "冻结基线(477表)": BL,
    "G-01xG-07 并轨骨架(DRAFT)": SEM / "changes/_drafts/g01g07-merged-skeleton.md",
    "材料包 D6 补充": SEM / "sync/w16-review-materials.md",
    "Day6 md": W16 / "第16周-Day6-G07集合门转正×活事件回放×G01并轨骨架.md",
}
for name, p in deliverables.items():
    assert p.exists(), f"缺失: {p}"
    print(f"  OK {name}: {p.name}（{p.stat().st_size} B）")

head = BL.read_text(encoding="utf-8").splitlines()[:8]
assert any("0392e107" in l for l in head) and any("禁止随手改" in l for l in head)

summary = [
    "==== W16-D6 实验汇总 ====",
    "活事件回放    : RED，7 positive BROKEN + 7 leak BROKEN + 0 死登记，归因 pg:000227/000229 逐行",
    "突变排练      : 6/6 PASS（孤儿红/citing绿/死登记红/无关编辑零误伤/等量换血红/fail-closed红）",
    "citing 生命周期: 构造 change -> 7/7 OK -> GREEN（变化留痕，不是禁止变化）",
    "对照实验      : swap 检出率 计数门 0% vs 集合门 100%；reorder 双门 0%（无假阳性）",
    f"交付物        : {len(deliverables)} 件全在位",
    "==== 全部断言通过 ====",
]
print("\n".join(summary))